In [1]:
import pandas as pd
import sqlite3

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"

conn = sqlite3.connect(db_path)

In [4]:
query = """
SELECT DISTINCT
    item_code,
    item_name,
    unit
FROM lab_raw
ORDER BY item_code
"""

items = pd.read_sql_query(query, conn)

items.head()

,item_code,item_name,unit
0,0001,蛋白分画,NaN
1,0003,ＡＬＢＰＦ,%
2,0005,アルフア１,%
3,0007,アルフア２,%
4,0009,ベータ,%


In [5]:
items.shape

(886, 3)

In [6]:
import sqlite3
import pandas as pd

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"

conn = sqlite3.connect(db_path)

df = pd.read_sql("""
SELECT item_code, item_name
FROM lab_raw
""", conn)

df.head()

,item_code,item_name
0,0017,総蛋白
1,0027,ＧＯＴ
2,0029,ＧＰＴ
3,0031,ＬＤＨ
4,0033,ＣＰＫ


In [7]:
code_check = (
    df.groupby("item_code")["item_name"]
    .nunique()
    .reset_index()
)

code_check = code_check[code_check["item_name"] > 1]

code_check

,item_code,item_name
215,2293,2
350,3727,2


In [8]:
problem_codes = code_check["item_code"].tolist()

df[df["item_code"].isin(problem_codes)] \
    .sort_values(["item_code", "item_name"])


,item_code,item_name
241547,2293,アルドス．
421302,2293,アルドス．
421773,2293,アルドス．
423368,2293,アルドス．
680529,2293,アルドス．
...,...,...
4055959,3727,測定値
4061113,3727,測定値
4063535,3727,測定値
147985,3727,Ｓ／ＣＯ


In [9]:
name_check = (
    df.groupby("item_name")["item_code"]
    .nunique()
    .reset_index()
)

name_check = name_check[name_check["item_code"] > 1]

name_check

,item_name,item_code
5,アシナガ,2
7,アスペル,3
12,アニサキス,2
19,アルテル,3
20,アルド,2
...,...,...
582,Ｍｇ,2
589,ＮＭ濃度,2
635,Ｓ／ＣＯ,4
637,ＳＥＡ,2


In [10]:
df["item_name_norm"] = (
    df["item_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

code_check_norm = (
    df.groupby("item_code")["item_name_norm"]
    .nunique()
    .reset_index()
)

code_check_norm[
    code_check_norm["item_name_norm"] > 1
]

,item_code,item_name_norm
215,2293,2
350,3727,2


In [11]:
mapping = (
    df.groupby("item_code")["item_name"]
    .unique()
    .reset_index()
)

mapping.head(20)

,item_code,item_name
0,0001,[蛋白分画]
1,0003,[ＡＬＢＰＦ]
2,0005,[アルフア１]
3,0007,[アルフア２]
4,0009,[ベータ]
5,0011,[ガンマー]
6,0015,[Ａ／ＧＰＦ]
7,0017,[総蛋白]
8,0018,[アルブミン]
9,0019,[Ａ／Ｇ比]


In [12]:
mapping["n_names"] = mapping["item_name"].apply(len)

mapping[mapping["n_names"] > 1]


,item_code,item_name,n_names
215,2293,"[アルドス．, ＲＩ相当値]",2
350,3727,"[Ｓ／ＣＯ, 測定値]",2


In [13]:
unit_check = pd.read_sql("""
SELECT item_code,
       item_name,
       unit,
       COUNT(*) as n
FROM lab_raw
GROUP BY item_code, unit
ORDER BY item_code
""", conn)

unit_check

,item_code,item_name,unit,n
0,0001,蛋白分画,NaN,51
1,0003,ＡＬＢＰＦ,%,51
2,0005,アルフア１,%,51
3,0007,アルフア２,%,51
4,0009,ベータ,%,51
...,...,...,...,...
880,8876,テイコプラ,MCG/ML,3
881,8880,ボリコナゾ,MCG/ML,2
882,9900,２４Ｈ蓄尿,ML,3
883,9998,乳濁,NaN,2722


In [14]:
unit_problem = (
    unit_check.groupby("item_code")["unit"]
    .nunique(dropna=False)
    .reset_index()
)

unit_problem = unit_problem[
    unit_problem["unit"] > 1
]

unit_problem

,item_code,unit
24,0051,2
183,2070,2
184,2071,2
185,2072,2
186,2073,2
187,2074,2
227,2373,2
235,2427,2
236,2429,2
237,2431,2


In [15]:
problem_units = pd.read_sql("""
SELECT item_code,
       item_name,
       unit,
       COUNT(*) as n
FROM lab_raw
WHERE item_code IN (
    '0051','2070','2071','2072','2073',
    '2074','2373','2427','2429','2431',
    '2433','3252','3579','3727','3739',
    '4832','4850','4852','5105','5534',
    '7519','7539','7541','7543'
)
GROUP BY item_code, item_name, unit
ORDER BY item_code
""", conn)

problem_units

,item_code,item_name,unit,n
0,0051,リパーゼ,IU/L,22
1,0051,リパーゼ,U/L,95
2,2070,ＦＡ－４Ｆ,NaN,1
3,2070,ＦＡ－４Ｆ,MCG/ML,7
4,2071,ＤＨＬＡ,NaN,7
5,2071,ＤＨＬＡ,MCG/ML,1
6,2072,ＡＡ,NaN,7
7,2072,ＡＡ,MCG/ML,1
8,2073,ＥＰＡ,NaN,7
9,2073,ＥＰＡ,MCG/ML,1


In [16]:
df_value = pd.read_sql("""
SELECT item_code, item_name, value
FROM lab_raw
LIMIT 1000
""", conn)

df_value.head(20)

DatabaseError: Execution failed on sql '
SELECT item_code, item_name, value
FROM lab_raw
LIMIT 1000
': no such column: value

In [17]:
pd.read_sql("""
PRAGMA table_info(lab_raw)
""", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,raw_id,INTEGER,0,None,1
1,1,Patient_ID,TEXT,0,None,0
2,2,sample_date,TEXT,0,None,0
3,3,item_name,TEXT,0,None,0
4,4,item_code,TEXT,0,None,0
5,5,value_raw,TEXT,0,None,0
6,6,unit,TEXT,0,None,0
7,7,hl_flag,TEXT,0,None,0
8,8,ref_low,TEXT,0,None,0
9,9,ref_high,TEXT,0,None,0


In [18]:
df_value = pd.read_sql("""
SELECT item_code,
       item_name,
       value_raw,
       unit
FROM lab_raw
LIMIT 100
""", conn)

df_value

,item_code,item_name,value_raw,unit
0,0017,総蛋白,6.9,G/DL
1,0027,ＧＯＴ,22,U/L
2,0029,ＧＰＴ,20,U/L
3,0031,ＬＤＨ,243,U/L
4,0033,ＣＰＫ,88,U/L
...,...,...,...,...
95,7641,ＭＣＨ,30.8,PG
96,7643,ＭＣＨＣ,31.9,%
97,0017,総蛋白,6.5,G/DL
98,0027,ＧＯＴ,18,U/L


In [19]:
df_all["sample_date_parsed"] = pd.to_datetime(
    df_all["sample_date"],
    errors="coerce"
)

df_all["sample_date_parsed"].isna().mean()

NameError: name 'df_all' is not defined

In [20]:
df_all = pd.read_sql("""
SELECT *
FROM lab_raw
""", conn)

In [21]:
df_all["sample_date_parsed"] = pd.to_datetime(
    df_all["sample_date"],
    errors="coerce"
)

In [22]:
df_all["sample_date_parsed"].isna().mean()

np.float64(0.0)

In [23]:
df_all[[
    "sample_date",
    "sample_date_parsed"
]].head(20)

,sample_date,sample_date_parsed
0,2018-02-05,2018-02-05
1,2018-02-05,2018-02-05
2,2018-02-05,2018-02-05
3,2018-02-05,2018-02-05
4,2018-02-05,2018-02-05
5,2018-02-05,2018-02-05
6,2018-02-05,2018-02-05
7,2018-02-05,2018-02-05
8,2018-02-05,2018-02-05
9,2018-02-05,2018-02-05


In [24]:
dup_check = (
    df_all.groupby([
        "Patient_ID",
        "sample_date",
        "item_code"
    ])
    .size()
    .reset_index(name="n")
)

dup_check[dup_check["n"] > 1]

,Patient_ID,sample_date,item_code,n
0,150001,2015-04-03,0017,2
1,150001,2015-04-03,0027,2
2,150001,2015-04-03,0029,2
3,150001,2015-04-03,0031,2
4,150001,2015-04-03,0033,2
...,...,...,...,...
2085009,251835,2025-11-13,7553,2
2085010,251835,2025-11-13,7555,2
2085011,251835,2025-11-13,7639,2
2085012,251835,2025-11-13,7641,2


In [25]:
dup_ids = dup_check[dup_check["n"] > 1]

df_dup = df_all.merge(
    dup_ids[
        ["Patient_ID", "sample_date", "item_code"]
    ],
    on=["Patient_ID", "sample_date", "item_code"]
)

df_dup.sort_values([
    "Patient_ID",
    "sample_date",
    "item_code"
])

,raw_id,Patient_ID,sample_date,item_name,item_code,value_raw,unit,hl_flag,ref_low,ref_high,comment,lab_company,facility,source_file,imported_at,sample_date_parsed
1556,1700,150001,2015-04-03,総蛋白,0017,5.9,G/DL,L,6.7,8.3,None,昭和メディカルサイエンス,居宅,DS150404.csv,2026-05-26 14:50:45,2015-04-03
1846,1999,150001,2015-04-03,総蛋白,0017,5.9,G/DL,L,6.7,8.3,None,昭和メディカルサイエンス,居宅,DS150405.csv,2026-05-26 14:50:45,2015-04-03
1557,1701,150001,2015-04-03,ＧＯＴ,0027,16,U/L,NaN,10,40,None,昭和メディカルサイエンス,居宅,DS150404.csv,2026-05-26 14:50:45,2015-04-03
1847,2000,150001,2015-04-03,ＧＯＴ,0027,16,U/L,NaN,10,40,None,昭和メディカルサイエンス,居宅,DS150405.csv,2026-05-26 14:50:45,2015-04-03
1558,1702,150001,2015-04-03,ＧＰＴ,0029,15,U/L,NaN,5,45,None,昭和メディカルサイエンス,居宅,DS150404.csv,2026-05-26 14:50:45,2015-04-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3668712,4176723,251835,2025-11-13,ＭＣＶ,7639,94.6,FL,NaN,79.0,100.0,None,昭和メディカルサイエンス,外来,DS251115.csv,2026-05-26 14:51:10,2025-11-13
3667060,4174788,251835,2025-11-13,ＭＣＨ,7641,31.7,PG,NaN,26.3,34.3,None,昭和メディカルサイエンス,外来,DS251114.csv,2026-05-26 14:51:10,2025-11-13
3668713,4176724,251835,2025-11-13,ＭＣＨ,7641,31.7,PG,NaN,26.3,34.3,None,昭和メディカルサイエンス,外来,DS251115.csv,2026-05-26 14:51:10,2025-11-13
3667061,4174789,251835,2025-11-13,ＭＣＨＣ,7643,33.5,%,NaN,30.7,36.6,None,昭和メディカルサイエンス,外来,DS251114.csv,2026-05-26 14:51:10,2025-11-13


In [26]:
dup_exact = df_all.duplicated(
    subset=[
        "Patient_ID",
        "sample_date",
        "item_code",
        "value_raw"
    ]
).mean()

dup_exact

np.float64(0.5011679519438408)

In [30]:
dup_n = df_all.duplicated(
    subset=[
        "Patient_ID",
        "sample_date",
        "item_code",
        "value_raw"
    ]
).sum()

dup_n

np.int64(2095938)

In [31]:
df_clean = df_all.drop_duplicates(
    subset=[
        "Patient_ID",
        "sample_date",
        "item_code",
        "value_raw",
        "unit"
    ]
)

In [32]:
dup_exact = df_all.duplicated(
    subset=[
        "Patient_ID",
        "sample_date",
        "item_code",
        "value_raw"
    ]
).mean()

dup_exact

np.float64(0.5011679519438408)

In [33]:
dup_n = df_all.duplicated(
    subset=[
        "Patient_ID",
        "sample_date",
        "item_code",
        "value_raw"
    ]
).sum()

dup_n

np.int64(2095938)

In [34]:
dup_mask = df_all.duplicated(
    subset=[
        "Patient_ID",
        "sample_date",
        "item_code",
        "value_raw",
        "unit"
    ],
    keep=False
)

df_dup = df_all[dup_mask]

df_dup.shape

(3672219, 16)

In [35]:
df_dup.groupby([
    "Patient_ID",
    "sample_date",
    "item_code",
    "value_raw"
])["source_file"].unique().head(20)

Patient_ID  sample_date  item_code  value_raw
150001      2015-04-03   0017       5.9          [DS150404.csv, DS150405.csv]
                         0027       16           [DS150404.csv, DS150405.csv]
                         0029       15           [DS150404.csv, DS150405.csv]
                         0031       244          [DS150404.csv, DS150405.csv]
                         0033       35           [DS150404.csv, DS150405.csv]
                         0037       20           [DS150404.csv, DS150405.csv]
                         0041       159          [DS150404.csv, DS150405.csv]
                         0047       199          [DS150404.csv, DS150405.csv]
                         0055       0.42         [DS150404.csv, DS150405.csv]
                         0063       14.8         [DS150404.csv, DS150405.csv]
                         0069       0.64         [DS150404.csv, DS150405.csv]
                         0081       4.7          [DS150404.csv, DS150405.csv]
                  

In [36]:
multi_value = (
    df_all.groupby([
        "Patient_ID",
        "sample_date",
        "item_code"
    ])["value_raw"]
    .nunique()
    .reset_index(name="n_values")
)

multi_value[
    multi_value["n_values"] > 1
]

,Patient_ID,sample_date,item_code,n_values
4814,150009,2016-05-26,7525,2
4815,150009,2016-05-26,7527,2
4816,150009,2016-05-26,7529,2
4817,150009,2016-05-26,7531,2
4818,150009,2016-05-26,7533,2
...,...,...,...,...
2061624,250190,2025-08-12,7533,2
2061625,250190,2025-08-12,7535,2
2061632,250190,2025-08-12,7639,2
2061633,250190,2025-08-12,7641,2


In [37]:
pd.read_sql("""
SELECT
    Patient_ID,
    sample_date,
    item_code,
    item_name,
    value_raw,
    unit,
    source_file
FROM lab_raw
WHERE Patient_ID = '150009'
AND sample_date = '2016-05-26'
AND item_code = '7525'
""", conn)

,Patient_ID,sample_date,item_code,item_name,value_raw,unit,source_file
0,150009,2016-05-26,7525,ＢＡＳ,0.0,%,DS160527.csv
1,150009,2016-05-26,7525,ＢＡＳ,0,%,DS160528.コピー.csv


In [38]:
multi_value_num.shape

NameError: name 'multi_value_num' is not defined

In [1]:
import sqlite3
import pandas as pd

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"
conn = sqlite3.connect(db_path)

# DBに存在するテーブル一覧
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("=== テーブル一覧 ===")
print(tables)

# Background_summaryのmemoサンプル
df = pd.read_sql("SELECT patient_ID, memo FROM Background_summary LIMIT 5", conn)
print("\n=== Background_summary のmemoサンプル ===")
print(df)

conn.close()

=== テーブル一覧 ===
                    name
0         Patient_Master
1             first_diag
2                  event
3   intervention_history
4       unexpected_death
5     Background_summary
6             unex_study
7           Freedocument
8       study_id_linkage
9                lab_raw
10       sqlite_sequence
11             lab_clean

=== Background_summary のmemoサンプル ===
   Patient_ID                                               memo
0      230344  ☆旧字体のため書類ボックス確認ください。\n・難聴あり、左補聴器\n・マイナ保険証スキャン年...
1      250004  ☆旧字体のため書類ボックス確認ください。\n・アレルギー：ニューキノロン系、チーズ\n・若い...
2      250251  時間　　：　　～　　：　　　　　　次回　　月　　日　　　日後　　：　　\n　　　　　　　　　...
3      190192  サムスカ導入(2021/1)\n・難病医療証あり！前月と当月ページ写真撮影(訪問時毎回)\n...
4      230381  　\n・本氏がんのこと知っている\n・P注意\n・左上腕シャントあり（BP採血右上肢で）\n...


In [2]:
import sqlite3
import pandas as pd

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"
csv_path = "/Users/muna/Hana_research/data/raw/NowSamari/patient_data_20260416.csv"

# CSVのmemo
df_csv = pd.read_csv(csv_path, encoding="cp932", low_memory=False)
df_csv = df_csv[["患者ID", "重要メモ"]].rename(columns={"患者ID": "patient_ID", "重要メモ": "memo"})

# DBのmemo
conn = sqlite3.connect(db_path)
df_db = pd.read_sql("SELECT patient_ID, memo FROM Background_summary WHERE patient_ID IS NOT NULL", conn)
conn.close()

# 型を合わせてマージ
df_csv["patient_ID"] = df_csv["patient_ID"].astype(str)
df_db["patient_ID"] = df_db["patient_ID"].astype(str)

df_merged = df_db.merge(df_csv, on="patient_ID", suffixes=("_db", "_csv"))

# 不一致の行だけ表示
mismatch = df_merged[df_merged["memo_db"] != df_merged["memo_csv"]]
print(f"不一致件数: {len(mismatch)}")
print(mismatch.head(10))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/muna/Hana_research/data/raw/NowSamari/patient_data_20260416.csv'

In [3]:
ls /Users/muna/Hana_research/data/raw/NowSamari/

csvoutput.py                        patient_data(20251116).csv
csvoutput説明.txt                   patient_data_20260419.csv
patient_data (6)_filtered.csv       patient_data_20260419_filtered.csv


In [4]:
import sqlite3
import pandas as pd

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"
conn = sqlite3.connect(db_path)

# カラム名確認
df = pd.read_sql("SELECT * FROM Background_summary LIMIT 3", conn)
conn.close()

print(df.columns.tolist())
print(df.head(3))

['Patient_ID', 'PS', 'visit_place', 'facility', 'JABC', 'ninchido', 'Kaiyodo', 'memo', 'tag', 'Study_ID']
   Patient_ID       PS visit_place       facility JABC ninchido Kaiyodo  \
0      230344  Grade 2          居宅             自宅   J2       Ⅱa    要介護1   
1      250004  Grade 1          居宅             自宅   J1       なし    要支援1   
2      250251  Grade 4          施設  ライフコミューン武蔵小杉＊   C2        Ⅳ    要介護4   

                                                memo            tag Study_ID  
0  ☆旧字体のため書類ボックス確認ください。\n・難聴あり、左補聴器\n・マイナ保険証スキャン年...  同一患家 区分２ ★マイナ  P004056  
1  ☆旧字体のため書類ボックス確認ください。\n・アレルギー：ニューキノロン系、チーズ\n・若い...  同一患家 区分２ ★マイナ  P004833  
2  時間　　：　　～　　：　　　　　　次回　　月　　日　　　日後　　：　　\n　　　　　　　　　...            NaN  P005051  


In [5]:
# 不一致の中身を詳しく確認
print(mismatch[["Patient_ID", "memo_db", "memo_csv"]].head(3).to_string())

NameError: name 'mismatch' is not defined

In [8]:
import sqlite3
import pandas as pd

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"
csv_path = "/Users/muna/Hana_research/data/raw/NowSamari/patient_data_20260419.csv"

# CSVのmemo
df_csv = pd.read_csv(csv_path, encoding="cp932", low_memory=False)
df_csv = df_csv[["患者ID", "重要メモ"]].rename(columns={"患者ID": "Patient_ID", "重要メモ": "memo"})

# DBのmemo
conn = sqlite3.connect(db_path)
df_db = pd.read_sql("SELECT Patient_ID, memo FROM Background_summary WHERE Patient_ID IS NOT NULL", conn)
conn.close()

# 型を合わせてマージ
df_csv["Patient_ID"] = df_csv["Patient_ID"].astype(str)
df_db["Patient_ID"] = df_db["Patient_ID"].astype(str)

df_merged = df_db.merge(df_csv, on="Patient_ID", suffixes=("_db", "_csv"))

# 不一致の行だけ表示
mismatch = df